In [ ]:
#| default_exp pythons

In [ ]:
#| export
from __future__ import annotations
import os, sys
from shutil import which
from fastcore.all import L, Path, first

In [ ]:
#| export
#: Directory names a project keeps its virtual environment in, best first.
VENV_DIRS = ('.venv', 'venv', 'env', '.env')
#: Where the interpreter sits inside one, on either platform.
VENV_BINS = ('bin/python', 'Scripts/python.exe')
#: The interpreters `nearest_python` looks for, as paths relative to a project folder.
VENV_PYTHONS = tuple(f'{d}/{b}' for d in VENV_DIRS for b in VENV_BINS)

#: Names a frozen host sets to point an interpreter at its own bundle.
BUNDLE_ONLY = ('PYTHONHOME', 'PYTHONPATH', 'PYTHONEXECUTABLE', '__PYVENV_LAUNCHER__', 'RESOURCEPATH')

In [ ]:
#| export
def strip_bundle(env, frozen=None):
    """`env` without a frozen host's interpreter redirection, unchanged where there is none.

    py2app and py2exe point `PYTHONHOME` and `PYTHONPATH` at the bundle so its own helper starts.
    A child that keeps them imports the bundle's standard library under another interpreter and
    fails somewhere that names nothing to do with the cause. Outside a bundle this returns what it
    was given, untouched, so a caller that claimed nothing about an environment still claims nothing.
    """
    if not (getattr(sys, 'frozen', False) if frozen is None else frozen): return env
    for name in BUNDLE_ONLY: env.pop(name, None)
    env['PYTHONUTF8'] = '1'
    return env

def clean_env():
    "This process's environment, safe to hand to a child."
    return strip_bundle(os.environ.copy(), frozen=True)

In [ ]:
#| export
def venv_env(python=None, env=None):
    "`env` (this process's, by default) with `python`'s virtual environment in front of it."
    env = strip_bundle(dict(os.environ if env is None else env))
    if not python: return env
    bindir = str(Path(python).parent)
    env['VIRTUAL_ENV'] = str(Path(bindir).parent)
    # Not `UV_PROJECT_ENVIRONMENT`. It is read wherever the process ends up rather than where it
    # started, so a `uv sync` run in another checkout syncs that project's lock into this venv and
    # prunes everything the lock does not name. uv finds the right environment from the directory.
    env.pop('UV_PROJECT_ENVIRONMENT', None)
    env['PATH'] = bindir + os.pathsep + env.get('PATH', '')
    env.pop('PYTHONHOME', None)
    return env

In [ ]:
#| export
def nearest_marked(start, markers, stop=None):
    """The nearest directory at or above `start` holding one of `markers`, and the marker it holds.

    `stop` is the folder the walk must not pass: what is above it belongs to something else.
    `markers` are relative paths, so `.venv/bin/python` asks about a file and `.git` a directory.
    """
    try: start = Path(start).resolve()
    except (OSError, ValueError): return None, None
    try: stop = Path(stop).resolve() if stop else None
    except (OSError, ValueError): stop = None
    for d in (start, *start.parents):
        for m in markers:
            if (hit := d/m).exists(): return d, hit
        if stop is not None and d == stop: break
    return None, None

In [ ]:
#| export
def _venv_pythons(root):
    "Every conventional venv interpreter path under `root`, as `(venv dir, path)`. Existence unchecked."
    root = Path(root)
    return L((d, root/d/b) for d in VENV_DIRS for b in VENV_BINS)

def project_python(roots=()):
    "The first conventional project-local virtualenv interpreter among `roots`, or None."
    for root in L(roots):
        if (p := first(p for _, p in _venv_pythons(root) if p.exists())): return str(p)
    return None

def nearest_python(start, stop=None):
    "The nearest conventional venv interpreter at or above `start`, not searched past `stop`."
    p = nearest_marked(start, VENV_PYTHONS, stop)[1]
    return str(p) if p else None

In [ ]:
#| export
def python_for(cwd=None, stop=None, roots=(), default=None):
    """The interpreter anything spawned for `cwd` should be inside.

    The walk up from `cwd` first, stopping at `stop` — the open folder, so a venv belonging to
    something above the workspace is not borrowed. Then whatever the caller nominates as its
    default, then the first venv among `roots`. None when nothing answers, which means "this
    interpreter" to everything downstream.
    """
    if cwd and (py := nearest_python(cwd, stop)): return py
    return default or project_python(roots)

In [ ]:
#| export
def _venv_roots(roots):
    "Each folder and the checkouts directly inside it: one folder of repos holds a venv each."
    for r in L(roots):
        yield Path(r)
        try: yield from sorted(d for d in Path(r).iterdir() if d.is_dir() and not d.name.startswith('.'))
        except OSError: pass

def find_pythons(roots=(), current=None, this_label='this one'):
    """Interpreters a kernel could be launched under: this one, `current`, and the venvs in reach.

    `current` is what `python_for` resolved for whatever is open, which the walk up can find deeper
    than a folder of checkouts is scanned. Listing it is what lets a picker mark it.
    """
    out, seen = L(), set()
    def add(p, label):
        p = str(p)
        if p in seen or not os.path.exists(p): return
        seen.add(p)
        out.append({'path': p, 'label': label})
    add(sys.executable, this_label)
    if current:
        d = Path(current).parents
        add(current, f'{d[2].name}/{d[1].name}' if len(d) > 2 else str(current))
    for r in _venv_roots(roots):
        for d, p in _venv_pythons(r): add(p, f'{Path(r).name}/{d}')
    for n in ('python3', 'python'):
        if (w := which(n)): add(w, f'{n} on PATH')
    return out